# Steam Game Price Tracker #


This AI tool is designed to help users to track steam game price

**In order to run this tool, you need to have a developer API from this site, "isthereanydeal.com" and store the API key into the .env file**




In [ ]:
import os
import sys
import json
import re
import requests
from openai import OpenAI
from dotenv import load_dotenv
import gradio as gr
import sqlite3
from datetime import datetime, timedelta



In [ ]:
# --- Configuration ---
load_dotenv(override=True)
ITAD_API_KEY = os.getenv('ITAD-API-KEY')


if not ITAD_API_KEY: 
    print("ITAD_API_KEY was not found in the env file. Please add the API key first and try again")
    sys.exit("Error: Condition was not met. Exiting script.")


In [ ]:
country_code_options = ['US', 'JP', 'GB', 'CA', 'AU', 'DE', 'FR']
selected_country = 'US'

def change_country_code(code):
    global selected_country
    selected_country = code
    print(f"country code updated: {selected_country}")
    return

In [ ]:
# Initialize your OpenAI client configured for your Llama 3.1 endpoint
# (e.g., Together AI, Groq, Fireworks, or a local vLLM/Ollama instance)

MODEL_API_KEY = os.getenv('MODEL_API_KEY')

if not MODEL_API_KEY: 
    print("MODEL_API_KEY was not found in the env file. Please add the API key first and try again")
    sys.exit("Error: Condition was not met. Exiting script.")

client = OpenAI(
    base_url="http://localhost:11434/v1", 
    api_key=MODEL_API_KEY
)

In [ ]:
DB_FILE = "steam_tracker.db"


In [ ]:
system_prompt = """You are a helpful gaming deal assistant that helps users track Steam game prices using an intelligent local caching system.

CRITICAL RULES FOR TOOL USE:
1. ONLY call the 'search_game' tool if the user explicitly mentions a specific video game title they want to check.
2. If the user says hello, asks general questions, or gives a vague query without a clear game title, DO NOT call any tools. Instead, reply politely and ask them what specific game they want to look up.
3. Never guess, hallucinate, or make up a game title if one isn't clearly provided in the chat history.

HOW TO HANDLE THE TOOL RESULT (check its "status" field):
- "match": the result has the game's price data. Summarize it as described below.
- "choices": the title matched more than one game. Show the "title" of each choice as a numbered list and ask which one they mean. Do not pick one yourself. When the user answers, call 'search_game' again with the exact title they picked.
- "not_found": tell the user no game matched and ask them to check the title.
- "error": tell the user the price lookup failed and to try again later.

HOW TO INTEGRATE AND REASON WITH THE TOOL DATA:
1. The tool automatically checks a local SQL 'game_library' database first. If the record is less than 24 hours old, it returns cached data to save API calls. Otherwise, it updates with live data.
2. When summarizing the deal, do not mention database implementation details (like SQLite or tables) to the end user unless they explicitly ask how the application works. 
3. Focus your summary on:
   - The current deal price vs the regular price (calculate the discount percentage if not already clear).
   - How close the current deal is to the historical lowest price (calculate the exact dollar difference to show value).
   - Provide a clear, analytical conclusion on whether they should buy now or wait for a better sale.
"""

In [ ]:
def get_price_by_id(game_id: str, official_title: str, country: str) -> dict:
    # -------------------------------------------------------------------------
    # STEP 1: Check the Local Cache
    # -------------------------------------------------------------------------
    print('Look up the title in the database...')
    with sqlite3.connect(DB_FILE) as conn:
        cursor = conn.cursor()
        cursor.execute(
            "SELECT current_deal, historical_lowest, last_search_date FROM game_library WHERE id = ? AND country_code = ?",
            (game_id, country)
        )
        row = cursor.fetchone()
        
    if row:
        print(f'game found in the db')
        cached_current, cached_lowest, last_search_str = row
        last_search_time = datetime.fromisoformat(last_search_str)
        
        # Check if the cache record is fresh (less than 24 hours old)
        if datetime.now() - last_search_time < timedelta(hours=24):
            print(f"⚡ Cache Hit! Loading '{official_title}' ({country}) data locally.")
            return {
                "title": official_title,
                "current_deal": json.loads(cached_current),
                "historical_lowest": json.loads(cached_lowest),
                "cached": True
            }

    # -------------------------------------------------------------------------
    # STEP 2: Cache Miss or Stale Data -> Query the API
    # -------------------------------------------------------------------------
    print(f"🌐 Cache Miss/Stale! Fetching fresh API data for '{official_title}' ({country}).")
    headers = {"ITAD-API-Key": ITAD_API_KEY}
    overview_url = "https://api.isthereanydeal.com/games/overview/v2"
    try:
        overview_res = requests.post(overview_url, headers=headers, params={"country": country}, json=[game_id])
        overview_res.raise_for_status()
        overview_data = overview_res.json()
        
        prices_list = overview_data.get("prices", [])
        if not prices_list:
            return {"error": f"No pricing details available for {official_title}."}
            
        game_deal_info = prices_list[0]
        current_deal = game_deal_info.get("current")
        historical_lowest = game_deal_info.get("lowest")
        now_str = datetime.now().isoformat()
        
        # -------------------------------------------------------------------------
        # STEP 3: Insert or Update the Database
        # -------------------------------------------------------------------------
        with sqlite3.connect(DB_FILE) as conn:
            print('Inserting or Updating data in the table')
            cursor = conn.cursor()
            cursor.execute("""
                INSERT INTO game_library (id, title, country_code, current_deal, historical_lowest, last_search_date)
                VALUES (?, ?, ?, ?, ?, ?)
                ON CONFLICT(id, country_code) DO UPDATE SET
                    current_deal = excluded.current_deal,
                    historical_lowest = excluded.historical_lowest,
                    last_search_date = excluded.last_search_date;
            """, (game_id, official_title, country, json.dumps(current_deal), json.dumps(historical_lowest), now_str))
            conn.commit()
            print("Done inserting/updating data")

        return {
            "title": official_title,
            "current_deal": current_deal,
            "historical_lowest": historical_lowest,
            "cached": False
        }
        
    except Exception as e:
        return {"error": f"Overview route failed: {str(e)}"}


In [ ]:
def normalize_title(title: str) -> str:
    # Lowercase, turn punctuation into spaces and collapse whitespace, so "Clair Obscur: Expedition 33"
    # and "clair obscur expedition 33" compare equal. Used for both stored titles and user queries.
    return " ".join(re.sub(r"[\W_]+", " ", title.casefold()).split())

In [ ]:
MAX_CHOICES = 10

# Every function below returns JSON with a "status" field:
#   "match"     -> price data for one game (same fields as get_price_by_id)
#   "choices"   -> {"choices": [{"id", "title"}, ...], "source": "catalog" | "api", "query"} for the user to pick from.
#                  With source "catalog" the list only covers games stored so far, so the UI should offer
#                  "None of these", which calls search_game again with search_online=True.
#   "not_found" -> nothing matched the query
#   "error"     -> {"error": "..."}

def price_response(game_id: str, official_title: str, country: str) -> str:
    result = get_price_by_id(game_id, official_title, country)
    status = "error" if "error" in result else "match"
    return json.dumps({"status": status, **result})


def choices_response(rows, source: str, query: str) -> str:
    return json.dumps({
        "status": "choices",
        "choices": [{"id": game_id, "title": title} for game_id, title in rows],
        "source": source,
        "query": query
    })


def get_game_price(official_title: str, country: str = "US") -> str:
    # Called once the user has picked a title, so it must already be in game_catalog
    with sqlite3.connect(DB_FILE) as conn:
        row = conn.execute("SELECT id FROM game_catalog WHERE official_title = ?", (official_title,)).fetchone()
    if not row:
        return json.dumps({"status": "error", "error": f"'{official_title}' is not in the game catalog. Search for it first."})
    return price_response(row[0], official_title, country)


def search_catalog(normalized_query: str) -> list:
    # Returns (id, official_title, is_exact) rows: exact title matches, or else titles containing the query
    print('Look up the title in the game_catalog...')
    with sqlite3.connect(DB_FILE) as conn:
        rows = conn.execute(
            "SELECT id, official_title FROM game_catalog WHERE normalized_title = ?",
            (normalized_query,)
        ).fetchall()
        if rows:
            return [(game_id, title, True) for game_id, title in rows]
        # normalize_title strips % and _, so the query can't add LIKE wildcards of its own
        rows = conn.execute(
            "SELECT id, official_title FROM game_catalog WHERE normalized_title LIKE ? ORDER BY length(official_title) LIMIT ?",
            (f"%{normalized_query}%", MAX_CHOICES)
        ).fetchall()
    return [(game_id, title, False) for game_id, title in rows]


def search_game(query: str, country: str = "US", search_online: bool = False) -> str:
    # search_online=True skips the local catalog, for when none of its choices were the right game
    normalized_query = normalize_title(query)
    if not normalized_query:
        return json.dumps({"status": "not_found", "query": query})

    # -------------------------------------------------------------------------
    # STEP 1: Look up the local catalog
    # -------------------------------------------------------------------------
    rows = []
    if not search_online:
        rows = search_catalog(normalized_query)
        # Exactly one exact match: no need to ask the user
        if len(rows) == 1 and rows[0][2]:
            game_id, official_title, _ = rows[0]
            print(f"exact match in game_catalog: {official_title}")
            return price_response(game_id, official_title, country)

    # Same title for several games, or partial matches: let the user pick
    if rows:
        print(f'{len(rows)} possible matches found in the game_catalog')
        return choices_response([(game_id, title) for game_id, title, _ in rows], "catalog", query)

    # -------------------------------------------------------------------------
    # STEP 2: Nothing stored yet, or the user asked to search online -> search the ITAD API
    # -------------------------------------------------------------------------
    headers = {"ITAD-API-Key": ITAD_API_KEY}
    search_url = "https://api.isthereanydeal.com/games/search/v1"
    try:
        print(f'No match found in the game_catalog for query: {query}')
        print(f'calling /search API for {query}')
        search_res = requests.get(search_url, headers=headers, params={"title": query})
        search_res.raise_for_status()
        search_data = search_res.json()
    except Exception as e:
        print(f"Search route failed: {str(e)}")
        return json.dumps({"status": "error", "error": f"Search route failed: {str(e)}"})

    if not search_data:
        return json.dumps({"status": "not_found", "query": query})

    # -------------------------------------------------------------------------
    # STEP 3: Store every result in game_catalog for future lookups
    # -------------------------------------------------------------------------
    now_str = datetime.now().isoformat()
    with sqlite3.connect(DB_FILE) as conn:
        conn.executemany(
            """
            INSERT INTO game_catalog (id, official_title, normalized_title, created_at)
            VALUES (?, ?, ?, ?)
            ON CONFLICT(id) DO NOTHING
            """,
            [(game["id"], game["title"], normalize_title(game["title"]), now_str) for game in search_data]
        )
        conn.commit()
    print(f"stored {len(search_data)} search results in game_catalog")

    # Exactly one result with the same title as the query: no need to ask the user
    exact = [game for game in search_data if normalize_title(game["title"]) == normalized_query]
    if len(exact) == 1:
        print(f"exact match in search results: {exact[0]['title']}")
        return price_response(exact[0]["id"], exact[0]["title"], country)

    return choices_response([(game["id"], game["title"]) for game in search_data[:MAX_CHOICES]], "api", query)

In [ ]:
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "search_game",
            "description": "Looks up a video game by title. If exactly one game matches, returns its current deal, regular pricing and historical all-time lowest price in the selected country. If several games match, returns their titles so the user can choose one.",
            "parameters": {
                "type": "object",
                "properties": {
                    "game_title": {
                        "type": "string",
                        "description": "The name of the video game (e.g., 'Cyberpunk 2077', 'Hades')."
                    }
                },
                "required": ["game_title"]
            }
        }
    }
]


def run_tool(tool_call) -> str:
    try:
        function_args = json.loads(tool_call.function.arguments)
    except json.JSONDecodeError:
        return json.dumps({"status": "error", "error": f"Invalid tool arguments: {tool_call.function.arguments}"})

    if tool_call.function.name == "search_game":
        # The country comes from the UI dropdown, not from the model
        return search_game(query=function_args.get("game_title", ""), country=selected_country)
    return json.dumps({"status": "error", "error": f"Unknown tool '{tool_call.function.name}'."})

In [ ]:
# =====================================================================
# 3. THE AGENT EXECUTION LOOP
# =====================================================================
def ask_llama(user_query: str):
    print(f"\nUser: {user_query}")
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_query}
    ]
    
    # First call: Send prompt and tools schema to Llama 3.1
    response = client.chat.completions.create(
        model="llama3.1", # Ensure this matches your exact hosted model name string
        messages=messages,
        tools=tools_schema,
        tool_choice="auto"
    )
    
    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls
    
    # Check if Llama 3.1 wants to run our API tool
    if tool_calls:
        print("🤖 Llama 3.1 decided to call the price tracker tool...")
        messages.append(response_message) # Append Llama's tool call intent to context
        
        for tool_call in tool_calls:
            # Execute our local Python code and send the JSON result back into the chat sequence
            messages.append({
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": tool_call.function.name,
                "content": run_tool(tool_call)
            })
        
        # Second call: Feed the API data back to Llama 3.1 so it can summarize
        second_response = client.chat.completions.create(
            model="llama3.1",
            messages=messages
        )
        print(f"🤖 Llama 3.1: {second_response.choices[0].message.content}")
    else:
        # If no tool was required (e.g., user just said "hello")
        print(f"🤖 Llama 3.1: {response_message.content}")



In [ ]:
# --- Run a Test ---
if __name__ == "__main__":
    ask_llama("Is Clair Obscur: Expedition 33 worth buying right now or should I wait?")

# Gradio UI

In [ ]:
# Key in an option value that means "search online for this query" instead of picking a game
SEARCH_ONLINE = "search_online"

def message_text(content):
    if isinstance(content, str):
        return content
    return "".join(part.get("text", "") for part in content if part.get("type") == "text")

def choices_message(result: dict) -> dict:
    # Gradio shows a message's "options" as buttons under it; clicking one calls handle_option_select.
    # Each value holds the game's id and title, so the click needs no other state.
    titles = [choice["title"] for choice in result["choices"]]
    options = []
    for i, choice in enumerate(result["choices"]):
        label = choice["title"]
        # Games can share a title, so number repeated titles to tell the buttons apart
        if titles.count(label) > 1:
            label = f"{label} ({titles[:i + 1].count(label)})"
        options.append({"label": label, "value": json.dumps({"id": choice["id"], "title": choice["title"]})})
    # The catalog only covers games searched before, so the right game may not be listed yet
    if result["source"] == "catalog":
        options.append({"label": "None of these, search online", "value": json.dumps({SEARCH_ONLINE: result["query"]})})
    return {
        "role": "assistant",
        "content": f"🔍 I found several games matching '{result['query']}'. Which one do you mean?",
        "options": options
    }

def finish_turn(messages, history, tool_outputs):
    # Several matching games: show them as options instead of letting the model write (and renumber) the list
    for output in tool_outputs:
        result = json.loads(output)
        if result.get("status") == "choices":
            history.append(choices_message(result))
            return history

    # Second call: Feed the API data back to Llama 3.1 so it can summarize
    second_response = client.chat.completions.create(
        model="llama3.1",
        messages=messages
    )
    print(f"🤖 Llama 3.1: {second_response.choices[0].message.content}")
    history.append({"role":"assistant", "content": second_response.choices[0].message.content})
    return history

def chat(message, history):
    if not message.strip():
        return "", history

    # Gradio 6 returns each message's content as a list of parts, e.g. [{"type": "text", "text": "..."}]
    history = [{"role":h["role"], "content":message_text(h["content"])} for h in history]
    history.append({"role":"user", "content":message})
    messages = [{"role": "system", "content": system_prompt}] + history
    # First call: Send prompt and tools schema to Llama 3.1
    response = client.chat.completions.create(
        model="llama3.1",
        messages=messages,
        tools=tools_schema,
    )
    
    response_message = response.choices[0].message
    print(response_message)
    tool_calls = response_message.tool_calls
    
    # Check if Llama 3.1 wants to run our API tool
    if tool_calls:
        print("🤖 Llama 3.1 decided to call the price tracker tool...")
        messages.append(response_message) # Append Llama's tool call intent to context
        
        tool_outputs = []
        for tool_call in tool_calls:
            # Execute our local Python code and send the JSON result back into the chat sequence
            tool_output = run_tool(tool_call)
            tool_outputs.append(tool_output)
            messages.append({
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": tool_call.function.name,
                "content": tool_output
            })

        return "", finish_turn(messages, history, tool_outputs)

    print(f"🤖 Llama 3.1: {response_message.content}")
    history.append({"role":"assistant", "content": response_message.content})
    return "", history

def handle_option_select(evt: gr.SelectData, chatbot_history):
    print(f"Option selected: {evt.value}")
    try:
        choice = json.loads(evt.value)
    except (TypeError, json.JSONDecodeError):
        return gr.skip()

    history = [{"role":h["role"], "content":message_text(h["content"])} for h in chatbot_history]
    if SEARCH_ONLINE in choice:
        game_title = choice[SEARCH_ONLINE]
        history.append({"role":"user", "content":"None of these"})
        tool_output = search_game(query=game_title, country=selected_country, search_online=True)
    else:
        game_title = choice["title"]
        history.append({"role":"user", "content":game_title})
        tool_output = price_response(choice["id"], game_title, selected_country)

    # Present the result to Llama 3.1 as a search_game call, the same way chat() does
    messages = [{"role": "system", "content": system_prompt}] + history + [
        {
            "role": "assistant",
            "content": "",
            "tool_calls": [{
                "id": "option_call",
                "type": "function",
                "function": {"name": "search_game", "arguments": json.dumps({"game_title": game_title})}
            }]
        },
        {"tool_call_id": "option_call", "role": "tool", "name": "search_game", "content": tool_output}
    ]
    return finish_turn(messages, history, [tool_output])


In [ ]:
initial_greeting = [
    {
        "role": "assistant", 
        "content": "👋 Hello! I am your Steam Price Tracking Assistant. Provide me with a game title, and I will instantly check its current deals and compare them to its historical all-time low to let you know if it's the right time to buy! Which game are we looking up today?"
    }
]

with gr.Blocks(title="Game Price Tracker AI") as ui:
    gr.Markdown('# Steam Game Price tracker AI Agent')
    with gr.Row():
        with gr.Column(scale=3, min_width=500):
            chatbot = gr.Chatbot(value=initial_greeting, height=400)
            with gr.Row():
                txt_input = gr.Textbox(show_label=False, placeholder="Ask about a game", scale=4)
                submit_btn = gr.Button('Send', variant="primary", scale=1)
        with gr.Column(scale=1):
            gr.Markdown('### Control & Watchlist Panel')
            with gr.Accordion("⚙️ Advanced Options", open=True):
                with gr.Column():
                    country_code_dropdown = gr.Dropdown(
                        choices=country_code_options, 
                        label="Select Country", 
                        value=country_code_options[0],
                        interactive=True  # Explicitly forces interactivity
                    )

    country_code_dropdown.change(fn=change_country_code, inputs=country_code_dropdown)
    txt_input.submit(fn=chat, inputs=[txt_input, chatbot], outputs=[txt_input, chatbot])
    submit_btn.click(
        fn=chat, 
        inputs=[txt_input, chatbot], 
        outputs=[txt_input, chatbot]
    )

    chatbot.option_select(fn=handle_option_select, inputs=[chatbot], outputs=[chatbot])

ui.launch()
        